# Joint-RPCA integration

**Purpose:** Integrate metabolomics and metagenomics feature tables using joint robust principal component analysis.

**Expected inputs**
- `../data/*platform*.csv`
- `../data/plasma_metadata_matched_main_outcomes.tsv`

**Main outputs**
- `../output_files/sig_metabolites_joint_rpca.csv`
- `../figures/*.png`

> Notes for reuse: data files are not included in this repository. Update paths in the cells below to match the local location of the approved, de-identified data release. Notebook outputs have been cleared for public sharing.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output_files'
FIGURE_DIR = PROJECT_ROOT / 'figures'

for directory in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import biom

from gemelli.rpca import feature_correlation_table, feature_covariance_table
from gemelli.rpca import joint_rpca, rpca
from skbio.stats.distance import permanova
from gemelli.preprocessing import matrix_rclr

#plotting
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#ignore warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
def ordination_scatterplots(rpca_results, metadata, x="PC1", y="PC2",
                            hue=None, hue_order=None, palette="tab10", 
                            markers=None, style=None, style_order=None, 
                            point_size=100, subplots=(2,3), figsize=(25, 10)):

    if subplots is not None:
        fig, axn = plt.subplots(subplots[0], subplots[1], figsize=figsize)
        axn = axn.flatten()
    else:
        fig = plt.figure(figsize=figsize)
        axn = [plt.gca()]

    for ax, (tblid, (ord_, _, _)) in zip(axn, rpca_results.items()):
        #prepare dataframe
        ord_samples = ord_.samples.rename(columns={0:'PC1', 1:'PC2', 
                                                   2:'PC3', 3:'PC4'})
        ord_plt = pd.concat([ord_samples, metadata], axis=1, sort=True)
        
        #plotting
        sns.scatterplot(x=x, y=y, hue=hue, hue_order=hue_order, 
                        palette=palette, style=style, 
                        style_order=style_order, markers=markers, 
                        data=ord_plt, s=point_size, ax=ax)
        ax.set_xlabel(x + ' (%.2f%%)' % (ord_.proportion_explained[0] * 100), color='black', weight='bold', fontsize=22)
        ax.set_ylabel(y + ' (%.2f%%)' % (ord_.proportion_explained[1] * 100), color='black', weight='bold', fontsize=22)
        ax.set_title(tblid, color='black', weight='bold', fontsize=22)
        # fix backround
        ax.set_facecolor('white')
        ax.set_axisbelow(True)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(True)
        ax.spines['bottom'].set_visible(True)
        ax.spines['top'].set_visible(False)
        for child in ax.get_children():
            if isinstance(child, matplotlib.spines.Spine):
                child.set_color('grey')

        for tick in ax.get_yticklabels():
            tick.set_fontproperties('arial')
            tick.set_weight('bold')
            tick.set_color("black")
            tick.set_fontsize(14)
        for tick in ax.get_xticklabels():
            tick.set_fontproperties('arial')
            tick.set_weight('bold')
            tick.set_color("black")
            tick.set_fontsize(14)
        ax.legend_.remove()
        
    plt.tight_layout()

    if subplots is not None:
        legend = ax.legend(loc=2, bbox_to_anchor=(-1.1, 2.6),
                           prop={'size':22}, title="",
                           fancybox=True, framealpha=.0,
                           ncol=len(hue_order), markerscale=3.5)
        legend.get_title().set_fontsize('16')
        # increase the line width in the legend 
        for line in legend.get_lines()[:]:
            line.set_linewidth(2.0)
        for line in legend.get_lines()[:]:
            line.set_linewidth(2.0)
    else:
        legend = ax.legend(loc=2, bbox_to_anchor=(1, 1),
                           prop={'size':18}, title="",
                           fancybox=True, framealpha=.0,
                           ncol=1, markerscale=3.5)
        legend.get_title().set_fontsize('16')
        # increase the line width in the legend 
        for line in legend.get_lines()[:]:
            line.set_linewidth(2.0)
        for line in legend.get_lines()[:]:
            line.set_linewidth(2.0)
        
    return ax


## Load data


In [ ]:
import qiime2 as q2

# Load Feature Tables

fts = {'Nightingale': pd.read_csv('/home/lakhatib/ADRC_plasma/data/Nightingale.csv', dtype={'sample_name': str}).set_index('sample_name'),
       'Metabolon': pd.read_csv('/home/lakhatib/ADRC_plasma/data/Metabolon.csv', dtype={'sample_name': str}).set_index('sample_name'),
       'UCSD': pd.read_csv('/home/lakhatib/ADRC_plasma/data/UCSD Known.csv', dtype={'sample_name': str}).set_index('sample_name'),
      }

In [ ]:
from sklearn.preprocessing import StandardScaler

fts_scaled = {}
for ft in fts.keys():
    df = fts[ft].copy()
    df[df.columns] = StandardScaler().fit_transform(df)
    fts_scaled[ft] = df

In [ ]:
md = pd.read_csv('../data/plasma_metadata_matched_main_outcomes.tsv', sep = '\t', dtype={'sample_name': str}).set_index('sample_name')


In [ ]:
sig = pd.read_csv('../output_files/sig_metab_metag_results.csv', index_col=0)


In [ ]:
sig['OGU'] = [taxonomy.split(';')[-1].strip() for taxonomy in sig['microbe_w_tax']]


In [ ]:
fts_sig = {}

for ft in fts_scaled.keys():
    if ft != 'Metagenomics':
        fts_sig[ft] = fts_scaled[ft][fts_scaled[ft].columns[fts_scaled[ft].columns.isin(sig['metabolite'])]]


## Joint-RPCA


In [ ]:
#transform tables in biom format
ft_biom = {}

for ft in fts:
    ft_biom[ft] = biom.Table(fts_sig[ft].values.T, 
                               sample_ids=fts_sig[ft].index, 
                               observation_ids=fts_sig[ft].columns)


In [ ]:
# ONLY RUN ON FIRST ITERATION
ord_jnt, dist_jnt, cv_jnt = joint_rpca([ft_biom['Metabolon'],ft_biom['UCSD']],
                                       n_test_samples=int(len(md.index) * 0.2), n_components=2,
                                      max_iterations=10, rclr_transform_tables=False)
joint_rpca_obj = {'joint': (ord_jnt, dist_jnt, cv_jnt)}


In [ ]:
joint_dm = pd.DataFrame(dist_jnt.data, index=dist_jnt.ids, columns=dist_jnt.ids)

joint_dm.to_csv('joint_dm.tsv', sep = '\t')

#plot CV error
plt.errorbar(x=joint_rpca_obj['joint'][2]['iteration'], 
             y=joint_rpca_obj['joint'][2]['mean_CV'], 
             yerr=joint_rpca_obj['joint'][2]['std_CV'],
             c='black')
plt.ylabel('CV-error', color='black')
plt.xlabel('iteration', color='black')
plt.yscale('log')
plt.show()

#check eigenvalues and proportion of variance explained
print('Eigenvalues:\n', joint_rpca_obj['joint'][0].eigvals)
print('Proportion of variance explained:\n', joint_rpca_obj['joint'][0].proportion_explained)


In [ ]:
#permanova
distances = joint_rpca_obj['joint'][1]
distances.ids = list(map(str, distances.ids))
md['Diagnosis'] = md['Diagnosis'].astype(str)

perm_res = permanova(distances, md.loc(axis=0)[distances.ids]['Diagnosis'])
perm_res


In [ ]:
# Extract the sample coordinates and feature loadings as DataFrames
samples_df = pd.DataFrame(ord_jnt.samples)
features_df = pd.DataFrame(ord_jnt.features)

pc_df = samples_df


In [ ]:
pc_df.to_csv('../output_files/sig_metabolites_joint_rpca.csv')


In [ ]:
palette = {'Cognitively Unimpaired': '#ff7f0e', 'Cognitively Impaired': '#1f77b4'}

pc_df['Diagnosis'] = md['Diagnosis']
pc_df = pc_df.dropna()
pc_df = pc_df[pc_df['Diagnosis'] != 'nan']
explained_variance_ratio = ord_jnt.proportion_explained*100

# Get top 10 features for PC1
top10_features_pc1 = features_df['PC1'].abs().sort_values(ascending=False).head(5).index
top10_features_pc2 = features_df['PC2'].abs().sort_values(ascending=False).head(5).index

# Combine top features
top_features = set(top10_features_pc1).union(set(top10_features_pc2))

features_df = features_df.loc[list(top_features)]

from skbio.stats.distance import permanova
from skbio.stats.distance import DistanceMatrix
from scipy.spatial.distance import pdist, squareform
from adjustText import adjust_text

def plot_biplot(group):
    # Set the Seaborn style to white to remove the gray grid background
    sns.set(style="white")

    # Assuming pca is your PCA object
    explained_variance_ratio = ord_jnt.proportion_explained * 100

    # Set background to white and the same color scheme as the second plot
    plt.figure(figsize=(5, 4))

    # Plot the principal components colored by groups
    sns.scatterplot(data=pc_df, x='PC1', y='PC2', hue=pc_df[group], palette=palette)

    # Label axes and add title
    plt.xlabel(f'Principal Component 1 ({round(explained_variance_ratio[0], 2)}%)')
    plt.ylabel(f'Principal Component 2 ({round(explained_variance_ratio[1], 2)}%)')
    plt.title(f'Joint-RPCA of Significant Plasma Metabolites Across Platforms')
    ax = plt.gca()
    handles, labels = ax.get_legend_handles_labels()

    # # Filter out unwanted label(s)
    # filtered = [(h, l) for h, l in zip(handles, labels) if l != "nan"]

    # Apply filtered legend
    ax.legend(loc='upper right')

    # Calculate PERMANOVA and get results
    permanova_result = permanova(distances, md.loc(axis=0)[distances.ids][group])
    test = permanova_result['test statistic']
    p = permanova_result['p-value']

    # Format p-value for display
    p_value_text = f"< .001" if p < 0.001 else f"= {p:.3f}".lstrip('0') if p < 0.01 else f"= {p:.2f}".lstrip('0')


    #Add PERMANOVA results box to the right of the plot
    plt.text(.02, .15, f"F = {test:.2f}\n$\mathit{{P}}$ value {p_value_text}", 
             fontsize=12, va='top', ha='left', transform=plt.gca().transAxes, bbox = dict(facecolor='white', alpha=0.7, edgecolor='gray'))

    # # Annotate top features with arrows and adjust text positions
    # texts = []
    # for feature in top_features:
    #     plt.arrow(0, 0, features_df.loc[feature, 'PC1'], features_df.loc[feature, 'PC2'],
    #               color='black', alpha=0.5, head_width=0.01, head_length=0.01)
    #     texts.append(
    #         plt.text(features_df.loc[feature, 'PC1'] * 1.2, features_df.loc[feature, 'PC2'] * 1.2,
    #                  feature, color='black', ha='center', va='center', fontsize=12)
    #     )

    # # Adjust text positions to avoid overlap
    # adjust_text(texts, arrowprops=dict(arrowstyle='->', color='white', alpha=0.5))

    # Save and show the plot
    plt.savefig(f'{group}_joint_rpca_sig_platforms.png', bbox_inches='tight')
    plt.show()


plot_biplot('Diagnosis')


In [ ]:
dfs = {}

for name, biom in ft_biom.items():
    dfs[name] = biom.to_dataframe(dense=True)
    
for name in dfs:
    dfs[name].index = [f"{name}_{f}" for f in dfs[name].index]
    
common_samples = set.intersection(
    *[set(df.columns) for df in dfs.values()]
)

print("Number of shared samples:", len(common_samples))

dfs_sub = [df[sorted(common_samples)] for df in dfs.values()]

merged_df = pd.concat(dfs_sub, axis=0)

from biom import Table

merged_biom = Table(
    merged_df.values,
    merged_df.index.tolist(),
    merged_df.columns.tolist()
)


In [ ]:
merged_df.to_csv('sig_metab_metag_table.csv')
